# LAB 3 — Auto Loader Initial Load

This notebook imports the shared Lab 3 configuration, validates the landing files, ingests the initial-schema JSON files incrementally with Auto Loader, and writes the data to the Bronze Delta table.

## 1. Load shared configuration

In [0]:
%run ./lab03_config

## 2. Display the initial-load configuration

In [0]:
initial_load_configuration = [
    ("Landing path", landing_path),
    ("Schema location", autoloader_schema_path),
    ("Checkpoint location", autoloader_checkpoint_path),
    ("Bronze table", file_bronze_table),
    ("Trigger type", trigger_type),
    ("Maximum files per trigger", str(max_files_per_trigger)),
    ("Schema evolution mode", schema_evolution_mode),
    ("Rescued data column", rescued_data_column),
]

display(
    spark.createDataFrame(
        initial_load_configuration,
        ["parameter", "value"]
    )
)

## 3. Validate the landing folder

In [0]:
landing_items = dbutils.fs.ls(landing_path)

landing_json_files = [
    file_info
    for file_info in landing_items
    if file_info.name.endswith(".json")
]

if not landing_json_files:
    raise FileNotFoundError(
        f"No JSON files were found in the landing path: {landing_path}"
    )

print(f"Landing JSON files: {len(landing_json_files)}")
display(landing_json_files[:20])

## 4. Define the Auto Loader source

The initial load uses schema inference, a persistent schema location, schema evolution settings, a rescued-data column, and a file limit per trigger.

In [0]:
from pyspark.sql.functions import col, current_timestamp

autoloader_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option(
        "cloudFiles.schemaLocation",
        autoloader_schema_path
    )
    .option(
        "cloudFiles.schemaEvolutionMode",
        schema_evolution_mode
    )
    .option(
        "cloudFiles.maxFilesPerTrigger",
        max_files_per_trigger
    )
    .option(
        "rescuedDataColumn",
        rescued_data_column
    )
    .option(
        "cloudFiles.inferColumnTypes",
        "true"
    )
    .load(landing_path)
    .withColumn(
        "_source_file",
        col("_metadata.file_path")
    )
    .withColumn(
        "_source_file_name",
        col("_metadata.file_name")
    )
    .withColumn(
        "_source_file_size",
        col("_metadata.file_size")
    )
    .withColumn(
        "_source_file_modification_time",
        col("_metadata.file_modification_time")
    )
    .withColumn(
        "_ingested_at",
        current_timestamp()
    )
)

print("Auto Loader source configured.")
autoloader_df.printSchema()

## 5. Start the Bronze write stream

In [0]:
writer = (
    autoloader_df.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        autoloader_checkpoint_path
    )
    .option("mergeSchema", "true")
    .queryName("lab03_autoloader_initial_load")
)

if trigger_type == "availableNow":
    query = (
        writer
        .trigger(availableNow=True)
        .toTable(file_bronze_table)
    )
elif trigger_type == "once":
    query = (
        writer
        .trigger(once=True)
        .toTable(file_bronze_table)
    )
elif trigger_type == "processingTime":
    query = (
        writer
        .trigger(processingTime="30 seconds")
        .toTable(file_bronze_table)
    )
else:
    raise ValueError(f"Unsupported trigger type: {trigger_type}")

print(f"Streaming query started: {query.name}")
print(f"Query ID: {query.id}")

## 6. Wait for the bounded initial load

`availableNow` and `once` finish automatically. A processing-time stream keeps running, so stop it manually after the initial files have been processed.

In [0]:
if trigger_type in {"availableNow", "once"}:
    query.awaitTermination()
    print("Initial Auto Loader query completed.")
else:
    print(
        "The processingTime stream is running. "
        "Monitor it and stop it manually when the initial load is complete."
    )

## 7. Inspect streaming progress

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    LongType,
    DoubleType,
)


def to_int(value):
    if value is None:
        return None

    try:
        return int(value)
    except (TypeError, ValueError):
        return None


def to_float(value):
    if value is None:
        return None

    try:
        return float(value)
    except (TypeError, ValueError):
        return None


progress_rows = []

for progress in query.recentProgress:
    sources = progress.get("sources") or []
    source = sources[0] if sources else {}
    source_metrics = source.get("metrics") or {}

    progress_rows.append(
        {
            "batch_id": to_int(
                progress.get("batchId")
            ),
            "num_input_rows": to_int(
                progress.get("numInputRows")
            ),
            "input_rows_per_second": to_float(
                progress.get("inputRowsPerSecond")
            ),
            "processed_rows_per_second": to_float(
                progress.get("processedRowsPerSecond")
            ),
            "trigger_execution_ms": to_int(
                (progress.get("durationMs") or {}).get(
                    "triggerExecution"
                )
            ),
            "num_files_outstanding": to_int(
                source_metrics.get(
                    "numFilesOutstanding"
                )
            ),
            "num_bytes_outstanding": to_int(
                source_metrics.get(
                    "numBytesOutstanding"
                )
            ),
        }
    )

progress_schema = StructType(
    [
        StructField(
            "batch_id",
            LongType(),
            True
        ),
        StructField(
            "num_input_rows",
            LongType(),
            True
        ),
        StructField(
            "input_rows_per_second",
            DoubleType(),
            True
        ),
        StructField(
            "processed_rows_per_second",
            DoubleType(),
            True
        ),
        StructField(
            "trigger_execution_ms",
            LongType(),
            True
        ),
        StructField(
            "num_files_outstanding",
            LongType(),
            True
        ),
        StructField(
            "num_bytes_outstanding",
            LongType(),
            True
        ),
    ]
)

if progress_rows:
    progress_df = spark.createDataFrame(
        progress_rows,
        schema=progress_schema
    )

    display(
        progress_df.orderBy("batch_id")
    )
else:
    print(
        "No recent progress is available "
        "in this notebook session."
    )

## 8. Validate the Bronze table

In [0]:
if not spark.catalog.tableExists(file_bronze_table):
    raise RuntimeError(
        f"Bronze table was not created: {file_bronze_table}"
    )

bronze_df = spark.table(file_bronze_table)

display(bronze_df.limit(20))
bronze_df.printSchema()

In [0]:
from pyspark.sql.functions import countDistinct, sum as spark_sum, when

validation_df = (
    bronze_df
    .agg(
        countDistinct("_source_file").alias("processed_source_files"),
        spark_sum(
            when(col(rescued_data_column).isNotNull(), 1).otherwise(0)
        ).alias("rescued_rows")
    )
)

total_bronze_rows = bronze_df.count()

print(f"Bronze rows: {total_bronze_rows:,}")
display(validation_df)

## 9. Compare landing files with processed files

In [0]:
processed_source_files = (
    bronze_df
    .select("_source_file")
    .distinct()
    .count()
)

landing_file_count = len(landing_json_files)

comparison_df = spark.createDataFrame(
    [
        ("Landing JSON files", landing_file_count),
        ("Processed source files", processed_source_files),
        ("Bronze rows", total_bronze_rows),
    ],
    ["metric", "value"]
)

display(comparison_df)

if processed_source_files > landing_file_count:
    print(
        "The Bronze table contains files from an earlier run or "
        "additional incremental batches."
    )
elif processed_source_files < landing_file_count:
    print(
        "Some landing files may still be unprocessed, or multiple files "
        "may contain no records."
    )
else:
    print("All current landing files are represented in Bronze.")

## 10. Inspect Auto Loader system locations

In [0]:
print("Schema location contents:")
display(dbutils.fs.ls(autoloader_schema_path))

print("Checkpoint location contents:")
display(dbutils.fs.ls(autoloader_checkpoint_path))

## 11. Final result

In [0]:
print("Lab 3 initial Auto Loader ingestion completed.")
print(f"Bronze table: {file_bronze_table}")
print(f"Bronze rows: {total_bronze_rows:,}")
print(f"Processed files: {processed_source_files}")
print("Next notebook: lab03_03_autoloader_schema_evolution")